In [ ]:
import pandas as pd
import folium
from geopy.distance import geodesic

from geopy.geocoders import Nominatim
import time
from folium.plugins import MarkerCluster
import os
import requests
from folium import IFrame

In [228]:
# 그룹별 최적 경로 찾는 함수
def optimize_route(df):
    unvisited = df.copy()
    visited = []
    current_point = unvisited.iloc[0]
    visited.append(current_point)
    unvisited = unvisited.drop(current_point.name)
    
    while not unvisited.empty:
        distances = unvisited.apply(
            lambda row: geodesic(
                (current_point['위도'], current_point['경도']),
                (row['위도'], row['경도'])
            ).meters,
            axis=1
        )
        nearest_idx = distances.idxmin()
        current_point = unvisited.loc[nearest_idx]
        visited.append(current_point)
        unvisited = unvisited.drop(nearest_idx)
    
    return pd.DataFrame(visited)

In [229]:
# 1. 각 코스 데이터 정의
course1 = pd.DataFrame({
    '장소명': ['북한산성', '보국문'],
    '위도': [37.645158972, 37.659024299],
    '경도': [126.974843478, 126.960877722]
})

course2 = pd.DataFrame({
    '장소명': ['정릉', '심우장', '성북구 한옥 거리'],
    '위도': [37.601191760879445, 37.593622925, 37.58911352646133],
    '경도': [127.00686314953727, 126.991659234, 127.00310169974995]
})

course3 = pd.DataFrame({
    '장소명': ['개운사', '흥천사', '아리랑고개'],
    '위도': [37.589988090, 37.598823596, 37.60143120683045],
    '경도': [127.028028351, 127.009246736, 127.01439608991056]
})

course4 = pd.DataFrame({
    '장소명': ['4.19 민주 묘지', '이준열사묘', '신익희 묘소'],
    '위도': [37.64901396176326, 37.642751018, 37.642372564],
    '경도': [127.0077077440808, 127.000985788, 127.001462283]
})

In [230]:
# ✅ 각각 최적화
path1_df = optimize_route(course1)
path2_df = optimize_route(course2)
path3_df = optimize_route(course3)
path4_df = optimize_route(course4)

In [231]:
# 2. 지도 생성
m = folium.Map(location=[37.61, 127.01], zoom_start=13)

# ✅ 지도 초기화
m = folium.Map(location=[path1_df.iloc[0]['위도'], path1_df.iloc[0]['경도']], zoom_start=13)


In [232]:
# ✅ 전체 마커 찍기
for idx, row in pd.concat([path1_df, path2_df, path3_df, path4_df]).iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        tooltip=row['장소명']
    ).add_to(m)

In [233]:
# ✅ 경로별 선 연결 (색 다르게)
folium.PolyLine(path1_df[['위도', '경도']].values.tolist(), color="blue", weight=5, opacity=0.7).add_to(m)
folium.PolyLine(path2_df[['위도', '경도']].values.tolist(), color="red", weight=5, opacity=0.7).add_to(m)
folium.PolyLine(path3_df[['위도', '경도']].values.tolist(), color="green", weight=5, opacity=0.7).add_to(m)
folium.PolyLine(path4_df[['위도', '경도']].values.tolist(), color="purple", weight=5, opacity=0.7).add_to(m)

In [ ]:
# 3. popup 구성
popup_html = """
<div style="font-size:13.5px;">
<b>📍 정릉</b><br>
🗺️ <b>주소: 서울특별시 성북구 정릉2동 산87-16번지<br>
👥 <b>최대 참여 인원:</b> 10명<br>
📖 <b>학습 내용:</b> A에 대한 내용 학습<br>
🫧 <b>문화 활동:</b> 플로깅
"""
iframe = IFrame(popup_html, width=250, height=100)
jeongneung_popup = folium.Popup(iframe, max_width=300)

# 마커 찍기 전용 반복문 (중복 없이)
popup_placed = False
for path_df in [path1_df, path2_df, path3_df, path4_df]:
    for idx, row in path_df.iterrows():
        if row['장소명'] == '정릉' and not popup_placed:
            folium.Marker(
                location=[row['위도'], row['경도']],
                tooltip='정릉',
                popup=jeongneung_popup
            ).add_to(m)
            popup_placed = True
        else:
            folium.Marker(
                location=[row['위도'], row['경도']],
                tooltip=row['장소명']
            ).add_to(m)



In [235]:
# 5. 저장
m.save('/Users/yoonjiwon/Desktop/full_course_map.html')